In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (StackingClassifier, StackingRegressor,
                              RandomForestClassifier, RandomForestRegressor,
                              ExtraTreesClassifier, ExtraTreesRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
from xgboost import XGBClassifier, XGBRegressor
import warnings
warnings.filterwarnings('ignore')



Libraries loaded!


In [ ]:
train = pd.read_csv('../data/credit_train.csv')

X = train.drop(['RiskTier', 'InterestRate'], axis=1)
y_cls = train['RiskTier']
y_reg = train['InterestRate']

print(f"Dataset shape: {X.shape}")
print(f"\nTask A — RiskTier distribution:\n{y_cls.value_counts().sort_index()}")
print(f"\nTask B — InterestRate summary:\n{y_reg.describe().round(2)}")

Dataset shape: (35000, 55)

Task A — RiskTier distribution:
RiskTier
0    6724
1    7283
2    6998
3    6812
4    7183
Name: count, dtype: int64

Task B — InterestRate summary:
count    35000.00
mean         7.31
std          4.19
min          4.99
25%          4.99
50%          6.08
75%          7.94
max         35.99
Name: InterestRate, dtype: float64


In [3]:
# Preprocessing is where most of the wins came from — model upgrades only mattered after we got this right.
# Asset/liability columns where NaN means "doesn't have one" → fill with 0, not median.
ZERO_FILL_COLS = [
    'PropertyValue',
    'MortgageOutstandingBalance',
    'StudentLoanOutstandingBalance',
    'AutoLoanOutstandingBalance',
    'VehicleValue',
    'InvestmentPortfolioValue',
    'SecondaryMonthlyIncome',
    'CollateralValue',
]

# Per EDA: only these two missingness patterns actually shift the target meaningfully.
MISSING_FLAG_COLS = ['InvestmentPortfolioValue', 'RevolvingUtilizationRate']

# Categoricals that show ~no signal in EDA (mean target barely varies across levels).
# EducationLevel removed: re-encoded ordinally instead.
DROP_COLS = ['State', 'JobCategory', 'MaritalStatus']

# Ordinal mapping — EDA shows monotonic RiskTier shift (2.17 → 1.82) across these levels.
EDUCATION_ORDER = {
    'No Diploma':   0,
    'HS Diploma':   1,
    'Some College': 2,
    'Bachelor':     3,
    'Master':       4,
    'PhD':          5,
}


def smart_preprocess(X_train, X_test=None):
    X_train = X_train.copy()
    if X_test is not None:
        X_test = X_test.copy()

    # 1. Informative missingness flags
    for col in MISSING_FLAG_COLS:
        if col in X_train.columns:
            X_train[f'{col}_isMissing'] = X_train[col].isnull().astype(int)
            if X_test is not None:
                X_test[f'{col}_isMissing']  = X_test[col].isnull().astype(int)

    # 2. CollateralType: explicit "None" for unsecured loans
    if 'CollateralType' in X_train.columns:
        X_train['CollateralType'] = X_train['CollateralType'].fillna('None')
        if X_test is not None:
            X_test['CollateralType'] = X_test['CollateralType'].fillna('None')

    # 3. EducationLevel → ordinal (preserve natural ordering)
    if 'EducationLevel' in X_train.columns:
        X_train['EducationLevel'] = X_train['EducationLevel'].map(EDUCATION_ORDER).fillna(1)
        if X_test is not None:
            X_test['EducationLevel'] = X_test['EducationLevel'].map(EDUCATION_ORDER).fillna(1)

    # 4. Asset/liability NaN → 0
    for col in ZERO_FILL_COLS:
        if col in X_train.columns:
            X_train[col] = X_train[col].fillna(0)
            if X_test is not None:
                X_test[col] = X_test[col].fillna(0)

    # 5. Engineered features echoing underwriter heuristics
    def _add_aggregates(df):
        late30 = df['NumberOfLatePayments30Days']
        late60 = df['NumberOfLatePayments60Days']
        late90 = df['NumberOfLatePayments90Days']
        charge = df['NumberOfChargeOffs']
        coll   = df['NumberOfCollections']
        bank   = df['NumberOfBankruptcies'].fillna(0)
        pubrec = df['NumberOfPublicRecords'].fillna(0)

        df['TotalLatePayments'] = late30 + late60 + late90
        df['DerogMarks']        = charge + coll + bank + pubrec

        # Fraction of accounts in good standing — true non-linear (division).
        df['SatisfactoryAccountRatio'] = (df['NumberOfSatisfactoryAccounts']
                                          / (df['NumberOfOpenAccounts'] + 1))

        # Domain-weighted severity (worse marks weighted higher — anti-FICO).
        df['RiskSeverityScore'] = (1*late30 + 3*late60 + 9*late90
                                   + 15*charge + 10*coll + 30*bank + 8*pubrec)
        return df

    X_train = _add_aggregates(X_train)
    if X_test is not None:
        X_test = _add_aggregates(X_test)

    # 6. Drop low-signal categoricals
    drop = [c for c in DROP_COLS if c in X_train.columns]
    X_train = X_train.drop(columns=drop)
    if X_test is not None:
        X_test = X_test.drop(columns=drop)

    # 7. Remaining numerics → median
    num_cols = X_train.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if X_train[col].isnull().any() or (X_test is not None and X_test[col].isnull().any()):
            med = X_train[col].median()
            X_train[col] = X_train[col].fillna(med)
            if X_test is not None:
                X_test[col] = X_test[col].fillna(med)

    # 8. Categoricals → mode + label encode
    cat_cols = X_train.select_dtypes(include=['object']).columns
    for col in cat_cols:
        mode = X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else 'Unknown'
        X_train[col] = X_train[col].fillna(mode)
        if X_test is not None:
            X_test[col] = X_test[col].fillna(mode)

        le = LabelEncoder()
        if X_test is not None:
            combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)])
            le.fit(combined)
            X_train[col] = le.transform(X_train[col].astype(str))
            X_test[col]  = le.transform(X_test[col].astype(str))
        else:
            X_train[col] = le.fit_transform(X_train[col].astype(str))

    if X_test is not None:
        return X_train, X_test
    return X_train


X_processed = smart_preprocess(X)
print(f"After preprocessing — shape: {X_processed.shape} | missing: {X_processed.isnull().sum().sum()}")
print(f"Engineered features: TotalLatePayments, DerogMarks, SatisfactoryAccountRatio, RiskSeverityScore")
print(f"EducationLevel     : ordinal-encoded (0=NoDiploma … 5=PhD)")
print(f"Missingness flags  : {sum(1 for c in X_processed.columns if c.endswith('_isMissing'))}")
print(f"Dropped noise cats : {DROP_COLS}")

After preprocessing — shape: (35000, 58) | missing: 0
Engineered features: TotalLatePayments, DerogMarks, SatisfactoryAccountRatio, RiskSeverityScore
EducationLevel     : ordinal-encoded (0=NoDiploma … 5=PhD)
Missingness flags  : 2
Dropped noise cats : ['State', 'JobCategory', 'MaritalStatus']


In [4]:
# 4-signal consensus matters — single-signal selection kept killing borderline-useful features.
# =====================================================================
# FEATURE SELECTION — multi-method consensus + correlation pruning
#
# Step 1: drop a feature if it scores low on ALL 4 signals
#   1a) XGBoost gain importance for RiskTier
#   1b) XGBoost gain importance for InterestRate
#   1c) Mutual information vs RiskTier
#   1d) Mutual information vs InterestRate
#
# Step 2: drop redundancy — for any pair of features with |corr| > 0.95,
#         drop whichever has the lower max_signal (keep the more useful one).
# =====================================================================

# Quick XGBoost importance for each task
imp_clf = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                        random_state=42, eval_metric='mlogloss',
                        tree_method='hist', n_jobs=-1).fit(X_processed, y_cls)
imp_reg = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                       random_state=42, tree_method='hist', n_jobs=-1).fit(X_processed, y_reg)

fi_clf = pd.Series(imp_clf.feature_importances_, index=X_processed.columns)
fi_reg = pd.Series(imp_reg.feature_importances_, index=X_processed.columns)

# Mutual information (non-linear univariate signal)
mi_clf = pd.Series(mutual_info_classif(X_processed, y_cls, random_state=42),
                   index=X_processed.columns)
mi_reg = pd.Series(mutual_info_regression(X_processed, y_reg, random_state=42),
                   index=X_processed.columns)

def norm(s):
    return s / s.max() if s.max() > 0 else s

scores = pd.DataFrame({
    'xgb_clf': norm(fi_clf),
    'xgb_reg': norm(fi_reg),
    'mi_clf':  norm(mi_clf),
    'mi_reg':  norm(mi_reg),
})
scores['max_signal'] = scores[['xgb_clf','xgb_reg','mi_clf','mi_reg']].max(axis=1)

# Step 1: signal-based dropping
THRESH = 0.05
NOISE_COLS = scores.index[scores['max_signal'] < THRESH].tolist()

print(f"=== Top 15 features by max signal ===")
print(scores.sort_values('max_signal', ascending=False).head(15)[
    ['xgb_clf','xgb_reg','mi_clf','mi_reg','max_signal']].round(3).to_string())

print(f"\n=== Step 1: dropped {len(NOISE_COLS)} low-signal features (< {THRESH:.0%} on all 4 signals) ===")
for c in NOISE_COLS:
    print(f"  - {c:38s}  max_signal = {scores.loc[c,'max_signal']:.3f}")

X_selected = X_processed.drop(columns=NOISE_COLS)

# Step 2: correlation pruning — drop the weaker partner of any near-duplicate pair
CORR_THRESH = 0.95
corr = X_selected.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

REDUNDANT_COLS = []
for col in upper.columns:
    partners = upper.index[upper[col] > CORR_THRESH].tolist()
    for p in partners:
        # keep the one with higher max_signal, drop the other
        if scores.loc[p, 'max_signal'] >= scores.loc[col, 'max_signal']:
            loser = col
        else:
            loser = p
        if loser not in REDUNDANT_COLS and loser in X_selected.columns:
            REDUNDANT_COLS.append(loser)

# Apply pruning (avoid dropping the same column twice if multiple pairs hit it)
REDUNDANT_COLS = [c for c in REDUNDANT_COLS if c in X_selected.columns]
X_selected = X_selected.drop(columns=REDUNDANT_COLS)

print(f"\n=== Step 2: dropped {len(REDUNDANT_COLS)} redundant features (|corr| > {CORR_THRESH}) ===")
for c in REDUNDANT_COLS:
    # find which feature(s) it duplicated
    duplicates_of = corr.index[(corr[c] > CORR_THRESH) & (corr.index != c)].tolist()
    duplicates_of = [d for d in duplicates_of if d in X_selected.columns]
    print(f"  - {c:38s}  redundant with: {duplicates_of}")

# Combine drops for the submission cell to use
DROPPED_COLS = NOISE_COLS + REDUNDANT_COLS
X_processed = X_selected
print(f"\nFinal feature count: {X_processed.shape[1]}  "
      f"(removed {len(NOISE_COLS)} noise + {len(REDUNDANT_COLS)} redundant = {len(DROPPED_COLS)} total)")

=== Top 15 features by max signal ===
                              xgb_clf  xgb_reg  mi_clf  mi_reg  max_signal
NumberOfLatePayments30Days      1.000    0.052   0.524   0.857       1.000
NumberOfChargeOffs              0.969    1.000   0.357   0.783       1.000
RiskSeverityScore               0.775    0.070   1.000   1.000       1.000
TotalLatePayments               0.391    0.609   0.561   0.957       0.957
RevolvingUtilizationRate        0.196    0.030   0.509   0.715       0.715
NumberOfLatePayments90Days      0.042    0.042   0.159   0.539       0.539
NumberOfLatePayments60Days      0.045    0.004   0.271   0.479       0.479
DerogMarks                      0.068    0.012   0.340   0.390       0.390
NumberOfBankruptcies            0.351    0.041   0.186   0.240       0.351
NumberOfCollections             0.335    0.033   0.315   0.349       0.349
NumberOfPublicRecords           0.260    0.003   0.034   0.028       0.260
AnnualIncome                    0.149    0.003   0.178   0.169

In [ ]:
# =====================================================================
# OPTUNA TUNING — every base learner for Task A (3-fold CV, TPE sampler)
# Each trial prints its params + score and the running best. The chosen
# best estimator for each slot is stored in `tuned_clf`.
# =====================================================================
import optuna
from sklearn.preprocessing import MinMaxScaler, RobustScaler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Per-model trial budgets — bigger for slow/important models, smaller for cheap ones.
N_TRIALS_CLF = {'xgb1': 25, 'xgb2': 25, 'rf': 15, 'et': 15, 'hgb': 20, 'mlp': 20}

SCALER_MAP = {'standard': StandardScaler, 'minmax': MinMaxScaler, 'robust': RobustScaler}


def run_study(name, objective, n_trials, scoring_label='acc'):
    """Run Optuna study; print each trial; return best_params."""
    print(f"\n>>> {name}  ({n_trials} trials)")
    def cb(study, trial):
        best = study.best_value
        ptxt = ", ".join(f"{k}={v}" for k, v in trial.params.items())
        if scoring_label == 'acc':
            print(f"  Trial {trial.number+1:>3}/{n_trials}  {ptxt[:80]:80s}  →  {trial.value*100:6.2f}%  (best={best*100:.2f}%)")
        else:
            print(f"  Trial {trial.number+1:>3}/{n_trials}  {ptxt[:80]:80s}  →  {trial.value:.4f}  (best={best:.4f})")
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, callbacks=[cb], show_progress_bar=False)
    print(f"  ★ BEST: {study.best_params}")
    return study.best_params


# ----- Objectives ---------------------------------------------------------
def obj_xgb1_clf(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 800, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 9),
        subsample=trial.suggest_float('subsample', 0.7, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.5, 5.0, log=True),
    )
    est = XGBClassifier(eval_metric='mlogloss', tree_method='hist',
                        random_state=42, n_jobs=-1, **params)
    return cross_val_score(est, X_processed, y_cls, cv=3, scoring='accuracy', n_jobs=-1).mean()

def obj_xgb2_clf(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 400, 1000, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        max_depth=trial.suggest_int('max_depth', 6, 10),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 0.9),
        reg_lambda=trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),
    )
    est = XGBClassifier(eval_metric='mlogloss', tree_method='hist',
                        random_state=7, n_jobs=-1, **params)
    return cross_val_score(est, X_processed, y_cls, cv=3, scoring='accuracy', n_jobs=-1).mean()

def obj_rf_clf(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 700, step=50),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
        max_features=trial.suggest_float('max_features', 0.3, 0.9),
        max_depth=trial.suggest_int('max_depth', 8, 30),
    )
    est = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    return cross_val_score(est, X_processed, y_cls, cv=3, scoring='accuracy', n_jobs=-1).mean()

def obj_et_clf(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 700, step=50),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
        max_features=trial.suggest_float('max_features', 0.3, 0.9),
        max_depth=trial.suggest_int('max_depth', 8, 30),
    )
    est = ExtraTreesClassifier(random_state=42, n_jobs=-1, **params)
    return cross_val_score(est, X_processed, y_cls, cv=3, scoring='accuracy', n_jobs=-1).mean()

def obj_hgb_clf(trial):
    params = dict(
        max_iter=trial.suggest_int('max_iter', 200, 800, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 10),
        l2_regularization=trial.suggest_float('l2_regularization', 0.0, 5.0),
        max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 15, 63),
    )
    est = HistGradientBoostingClassifier(random_state=42, **params)
    return cross_val_score(est, X_processed, y_cls, cv=3, scoring='accuracy', n_jobs=-1).mean()

def obj_mlp_clf(trial):
    scaler_name = trial.suggest_categorical('scaler', ['standard', 'minmax', 'robust'])
    arch = trial.suggest_categorical('arch', ['(128,)', '(128,64)', '(256,128)', '(256,128,64)'])
    alpha = trial.suggest_float('alpha', 1e-5, 1e-2, log=True)
    lr = trial.suggest_float('learning_rate_init', 5e-4, 5e-3, log=True)
    pipe = Pipeline([
        ('scaler', SCALER_MAP[scaler_name]()),
        ('mlp', MLPClassifier(hidden_layer_sizes=eval(arch), alpha=alpha,
                              learning_rate_init=lr, activation='relu',
                              max_iter=200, early_stopping=True,
                              n_iter_no_change=15, random_state=42)),
    ])
    return cross_val_score(pipe, X_processed, y_cls, cv=3, scoring='accuracy', n_jobs=-1).mean()


print("=" * 90)
print("CLASSIFIER OPTUNA TUNING — 6 base learners")
print("=" * 90)

best_params_clf = {}
best_params_clf['xgb1'] = run_study('xgb1_clf', obj_xgb1_clf, N_TRIALS_CLF['xgb1'])
best_params_clf['xgb2'] = run_study('xgb2_clf', obj_xgb2_clf, N_TRIALS_CLF['xgb2'])
best_params_clf['rf']   = run_study('rf_clf',   obj_rf_clf,   N_TRIALS_CLF['rf'])
best_params_clf['et']   = run_study('et_clf',   obj_et_clf,   N_TRIALS_CLF['et'])
best_params_clf['hgb']  = run_study('hgb_clf',  obj_hgb_clf,  N_TRIALS_CLF['hgb'])
best_params_clf['mlp']  = run_study('mlp_clf',  obj_mlp_clf,  N_TRIALS_CLF['mlp'])

# Build the tuned estimator for each slot from the chosen params
tuned_clf = {}
tuned_clf['xgb1'] = XGBClassifier(eval_metric='mlogloss', tree_method='hist',
                                  random_state=42, n_jobs=-1, **best_params_clf['xgb1'])
tuned_clf['xgb2'] = XGBClassifier(eval_metric='mlogloss', tree_method='hist',
                                  random_state=7, n_jobs=-1, **best_params_clf['xgb2'])
tuned_clf['rf']   = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params_clf['rf'])
tuned_clf['et']   = ExtraTreesClassifier(random_state=42, n_jobs=-1, **best_params_clf['et'])
tuned_clf['hgb']  = HistGradientBoostingClassifier(random_state=42, **best_params_clf['hgb'])

mlp_p = best_params_clf['mlp']
tuned_clf['mlp']  = Pipeline([
    ('scaler', SCALER_MAP[mlp_p['scaler']]()),
    ('mlp', MLPClassifier(hidden_layer_sizes=eval(mlp_p['arch']),
                          alpha=mlp_p['alpha'],
                          learning_rate_init=mlp_p['learning_rate_init'],
                          activation='relu', max_iter=200, early_stopping=True,
                          n_iter_no_change=15, random_state=42)),
])
mlp_clf_pipe = tuned_clf['mlp']  # backward-compat alias

print("\n" + "=" * 90)
print("All 6 classifier base learners tuned. tuned_clf ready for stacking.")
print("=" * 90)

CLASSIFIER OPTUNA TUNING — 6 base learners

>>> xgb1_clf  (25 trials)
  Trial   1/25  n_estimators=400, learning_rate=0.1358198535217987, max_depth=8, subsample=0.879  →   79.16%  (best=79.16%)
  Trial   2/25  n_estimators=200, learning_rate=0.11454791487656053, max_depth=7, subsample=0.91  →   79.31%  (best=79.31%)
  Trial   3/25  n_estimators=700, learning_rate=0.030678895924589226, max_depth=5, subsample=0.7  →   78.55%  (best=79.31%)
  Trial   4/25  n_estimators=450, learning_rate=0.03596444270828742, max_depth=7, subsample=0.74  →   79.43%  (best=79.43%)
  Trial   5/25  n_estimators=450, learning_rate=0.09729870597990298, max_depth=5, subsample=0.85  →   80.75%  (best=80.75%)
  Trial   6/25  n_estimators=550, learning_rate=0.028199996263578355, max_depth=4, subsample=0.9  →   74.52%  (best=80.75%)
  Trial   7/25  n_estimators=350, learning_rate=0.02435000636897718, max_depth=8, subsample=0.83  →   77.35%  (best=80.75%)
  Trial   8/25  n_estimators=200, learning_rate=0.124951379588

In [ ]:
print("=" * 60)
print("TASK A: RiskTier — Stacked Ensemble (6 tuned base + meta)  [5-fold CV]")
print("=" * 60)

base_clf = [(name, tuned_clf[name]) for name in ('xgb1', 'xgb2', 'rf', 'et', 'hgb', 'mlp')]

clf = StackingClassifier(
    estimators=base_clf,
    final_estimator=LogisticRegression(max_iter=2000, multi_class='multinomial',
                                       C=1.0, random_state=42),
    cv=3,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(clf, X_processed, y_cls, cv=skf, scoring='accuracy', n_jobs=-1)

for i, s in enumerate(cv_acc, 1):
    print(f"  Fold {i}: {s*100:.2f}%")

print("-" * 60)
print(f"  FINAL CV Accuracy:  {cv_acc.mean()*100:.2f}%  (+/- {cv_acc.std()*100:.2f}%)")
print("=" * 60)

TASK A: RiskTier — Stacked Ensemble (6 tuned base + meta)  [5-fold CV]
  Fold 1: 86.61%
  Fold 2: 89.00%
  Fold 3: 88.80%
  Fold 4: 88.46%
  Fold 5: 87.74%
------------------------------------------------------------
  FINAL CV Accuracy:  88.12%  (+/- 0.87%)


In [ ]:
# =====================================================================
# OOF RiskTier features for the regressor
# RiskTier is essentially a binned summary of credit-worthiness, and the
# strongest single driver of InterestRate. We give the regressor the
# classifier's accumulated knowledge by feeding 5 OOF class probabilities
# (one per tier) as additional features. Using out-of-fold predictions
# avoids any train→target leakage.
# =====================================================================
from sklearn.model_selection import cross_val_predict

print("Computing OOF RiskTier probabilities (5-fold) to feed into Task B...")

oof_clf = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6,
                        subsample=0.9, colsample_bytree=0.8, reg_lambda=1.0,
                        random_state=42, eval_metric='mlogloss',
                        tree_method='hist', n_jobs=-1)

oof_proba = cross_val_predict(oof_clf, X_processed, y_cls, cv=skf,
                              method='predict_proba', n_jobs=-1)

# Also include the soft "expected tier" — a single linear summary
expected_tier = oof_proba @ np.arange(5)

X_processed_reg = X_processed.copy()
for i in range(5):
    X_processed_reg[f'oof_riskTier_p{i}'] = oof_proba[:, i]
X_processed_reg['oof_expected_tier'] = expected_tier

print(f"X_processed shape    : {X_processed.shape}")
print(f"X_processed_reg shape: {X_processed_reg.shape}  (+ 5 tier probas + expected tier)")

Computing OOF RiskTier probabilities (5-fold) to feed into Task B...
X_processed shape    : (35000, 18)
X_processed_reg shape: (35000, 24)  (+ 5 tier probas + expected tier)


In [ ]:
# =====================================================================
# OPTUNA TUNING — every base learner for Task B (3-fold CV, TPE sampler)
# Tuned on the augmented X_processed_reg (with OOF tier features).
# =====================================================================
N_TRIALS_REG = {'xgb1': 25, 'xgb2': 25, 'rf': 15, 'et': 15, 'hgb': 20, 'mlp': 20}


def obj_xgb1_reg(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 800, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 9),
        subsample=trial.suggest_float('subsample', 0.7, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.5, 5.0, log=True),
    )
    est = XGBRegressor(tree_method='hist', random_state=42, n_jobs=-1, **params)
    return cross_val_score(est, X_processed_reg, y_reg, cv=3, scoring='r2', n_jobs=-1).mean()

def obj_xgb2_reg(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 400, 1000, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        max_depth=trial.suggest_int('max_depth', 6, 10),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 0.9),
        reg_lambda=trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),
    )
    est = XGBRegressor(tree_method='hist', random_state=7, n_jobs=-1, **params)
    return cross_val_score(est, X_processed_reg, y_reg, cv=3, scoring='r2', n_jobs=-1).mean()

def obj_rf_reg(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 700, step=50),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
        max_features=trial.suggest_float('max_features', 0.3, 0.9),
        max_depth=trial.suggest_int('max_depth', 8, 30),
    )
    est = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    return cross_val_score(est, X_processed_reg, y_reg, cv=3, scoring='r2', n_jobs=-1).mean()

def obj_et_reg(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 700, step=50),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
        max_features=trial.suggest_float('max_features', 0.3, 0.9),
        max_depth=trial.suggest_int('max_depth', 8, 30),
    )
    est = ExtraTreesRegressor(random_state=42, n_jobs=-1, **params)
    return cross_val_score(est, X_processed_reg, y_reg, cv=3, scoring='r2', n_jobs=-1).mean()

def obj_hgb_reg(trial):
    params = dict(
        max_iter=trial.suggest_int('max_iter', 200, 800, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 10),
        l2_regularization=trial.suggest_float('l2_regularization', 0.0, 5.0),
        max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 15, 63),
    )
    est = HistGradientBoostingRegressor(random_state=42, **params)
    return cross_val_score(est, X_processed_reg, y_reg, cv=3, scoring='r2', n_jobs=-1).mean()

def obj_mlp_reg(trial):
    scaler_name = trial.suggest_categorical('scaler', ['standard', 'minmax', 'robust'])
    arch = trial.suggest_categorical('arch', ['(128,)', '(128,64)', '(256,128)', '(256,128,64)'])
    alpha = trial.suggest_float('alpha', 1e-5, 1e-2, log=True)
    lr = trial.suggest_float('learning_rate_init', 5e-4, 5e-3, log=True)
    pipe = Pipeline([
        ('scaler', SCALER_MAP[scaler_name]()),
        ('mlp', MLPRegressor(hidden_layer_sizes=eval(arch), alpha=alpha,
                             learning_rate_init=lr, activation='relu',
                             max_iter=200, early_stopping=True,
                             n_iter_no_change=15, random_state=42)),
    ])
    return cross_val_score(pipe, X_processed_reg, y_reg, cv=3, scoring='r2', n_jobs=-1).mean()


print("=" * 90)
print("REGRESSOR OPTUNA TUNING — 6 base learners (on X_processed_reg)")
print("=" * 90)

best_params_reg = {}
best_params_reg['xgb1'] = run_study('xgb1_reg', obj_xgb1_reg, N_TRIALS_REG['xgb1'], scoring_label='r2')
best_params_reg['xgb2'] = run_study('xgb2_reg', obj_xgb2_reg, N_TRIALS_REG['xgb2'], scoring_label='r2')
best_params_reg['rf']   = run_study('rf_reg',   obj_rf_reg,   N_TRIALS_REG['rf'],   scoring_label='r2')
best_params_reg['et']   = run_study('et_reg',   obj_et_reg,   N_TRIALS_REG['et'],   scoring_label='r2')
best_params_reg['hgb']  = run_study('hgb_reg',  obj_hgb_reg,  N_TRIALS_REG['hgb'],  scoring_label='r2')
best_params_reg['mlp']  = run_study('mlp_reg',  obj_mlp_reg,  N_TRIALS_REG['mlp'],  scoring_label='r2')

tuned_reg = {}
tuned_reg['xgb1'] = XGBRegressor(tree_method='hist', random_state=42, n_jobs=-1, **best_params_reg['xgb1'])
tuned_reg['xgb2'] = XGBRegressor(tree_method='hist', random_state=7,  n_jobs=-1, **best_params_reg['xgb2'])
tuned_reg['rf']   = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params_reg['rf'])
tuned_reg['et']   = ExtraTreesRegressor(random_state=42, n_jobs=-1,   **best_params_reg['et'])
tuned_reg['hgb']  = HistGradientBoostingRegressor(random_state=42,    **best_params_reg['hgb'])

mlp_p = best_params_reg['mlp']
tuned_reg['mlp']  = Pipeline([
    ('scaler', SCALER_MAP[mlp_p['scaler']]()),
    ('mlp', MLPRegressor(hidden_layer_sizes=eval(mlp_p['arch']),
                         alpha=mlp_p['alpha'],
                         learning_rate_init=mlp_p['learning_rate_init'],
                         activation='relu', max_iter=200, early_stopping=True,
                         n_iter_no_change=15, random_state=42)),
])
mlp_reg_pipe = tuned_reg['mlp']

print("\n" + "=" * 90)
print("All 6 regressor base learners tuned. tuned_reg ready for stacking.")
print("=" * 90)

REGRESSOR OPTUNA TUNING — 6 base learners (on X_processed_reg)

>>> xgb1_reg  (25 trials)
  Trial   1/25  n_estimators=400, learning_rate=0.1358198535217987, max_depth=8, subsample=0.879  →  0.8140  (best=0.8140)
  Trial   2/25  n_estimators=200, learning_rate=0.11454791487656053, max_depth=7, subsample=0.91  →  0.8281  (best=0.8281)
  Trial   3/25  n_estimators=700, learning_rate=0.030678895924589226, max_depth=5, subsample=0.7  →  0.8359  (best=0.8359)
  Trial   4/25  n_estimators=450, learning_rate=0.03596444270828742, max_depth=7, subsample=0.74  →  0.8318  (best=0.8359)
  Trial   5/25  n_estimators=450, learning_rate=0.09729870597990298, max_depth=5, subsample=0.85  →  0.8313  (best=0.8359)
  Trial   6/25  n_estimators=550, learning_rate=0.028199996263578355, max_depth=4, subsample=0.9  →  0.8347  (best=0.8359)
  Trial   7/25  n_estimators=350, learning_rate=0.02435000636897718, max_depth=8, subsample=0.83  →  0.8310  (best=0.8359)
  Trial   8/25  n_estimators=200, learning_rate=0

In [ ]:
print("=" * 60)
print("TASK B: InterestRate — Stacked Ensemble (6 tuned base + meta)  [5-fold CV]")
print("=" * 60)

base_reg = [(name, tuned_reg[name]) for name in ('xgb1', 'xgb2', 'rf', 'et', 'hgb', 'mlp')]

reg = StackingRegressor(
    estimators=base_reg,
    final_estimator=Ridge(alpha=1.0, random_state=42),
    cv=3,
    passthrough=False,
    n_jobs=-1,
)

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2   = cross_val_score(reg, X_processed_reg, y_reg, cv=kf, scoring='r2',                          n_jobs=-1)
cv_rmse = -cross_val_score(reg, X_processed_reg, y_reg, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)

for i, (r, rm) in enumerate(zip(cv_r2, cv_rmse), 1):
    print(f"  Fold {i}:  R² = {r:.4f}  |  RMSE = {rm:.4f}%")

print("-" * 60)
print(f"  FINAL CV R²:    {cv_r2.mean():.4f}   (+/- {cv_r2.std():.4f})")
print(f"  FINAL CV RMSE:  {cv_rmse.mean():.4f}% (+/- {cv_rmse.std():.4f}%)")
print("=" * 60)

TASK B: InterestRate — Stacked Ensemble (6 tuned base + meta)  [5-fold CV]
  Fold 1:  R² = 0.8401  |  RMSE = 1.6292%
  Fold 2:  R² = 0.8336  |  RMSE = 1.6571%
  Fold 3:  R² = 0.8562  |  RMSE = 1.6558%
  Fold 4:  R² = 0.8299  |  RMSE = 1.6246%
  Fold 5:  R² = 0.8666  |  RMSE = 1.6418%
------------------------------------------------------------
  FINAL CV R²:    0.8453   (+/- 0.0140)
  FINAL CV RMSE:  1.6417% (+/- 0.0133%)


### Final fit on full training data + submission

In [ ]:
test = pd.read_csv('../data/credit_test.csv')
X_train_proc, X_test_proc = smart_preprocess(X, test)

# Drop the same noise + redundant features identified during selection
X_train_proc = X_train_proc.drop(columns=DROPPED_COLS)
X_test_proc  = X_test_proc.drop(columns=DROPPED_COLS)

# --- Task A: classifier ----------------------------------------------
clf.fit(X_train_proc, y_cls)
pred_cls   = clf.predict(X_test_proc)
test_proba = clf.predict_proba(X_test_proc)

# --- Task B: augment with OOF/test RiskTier probabilities ------------
X_train_reg = X_train_proc.copy()
for i in range(5):
    X_train_reg[f'oof_riskTier_p{i}'] = oof_proba[:, i]
X_train_reg['oof_expected_tier'] = oof_proba @ np.arange(5)

X_test_reg = X_test_proc.copy()
for i in range(5):
    X_test_reg[f'oof_riskTier_p{i}'] = test_proba[:, i]
X_test_reg['oof_expected_tier'] = test_proba @ np.arange(5)

reg.fit(X_train_reg, y_reg)
pred_reg = np.clip(reg.predict(X_test_reg), 4.99, 35.99).round(2)

submission = pd.DataFrame({
    'Id':           range(len(pred_cls)),
    'RiskTier':     pred_cls,
    'InterestRate': pred_reg,
})
submission.to_csv('submission.csv', index=False)

acc      = cv_acc.mean()
r2       = cv_r2.mean()
combined = 0.5 * acc + 0.5 * r2

print("\n" + "=" * 60)
print("FINAL CROSS-VALIDATION SCORES (Stacked ensemble + OOF tier)")
print("=" * 60)
print(f"  Features (clf):     {X_train_proc.shape[1]}")
print(f"  Features (reg):     {X_train_reg.shape[1]}  (+6 OOF features)")
print(f"  Task A — Accuracy:  {acc*100:.2f}%   (+/- {cv_acc.std()*100:.2f}%)")
print(f"  Task B — R²:        {r2:.4f}     (+/- {cv_r2.std():.4f})")
print(f"  Task B — RMSE:      {cv_rmse.mean():.4f}%   (+/- {cv_rmse.std():.4f}%)")
print("-" * 60)
print(f"  COMBINED (0.5·acc + 0.5·R²):  {combined:.4f}")
print("=" * 60)
print("\nSaved submission.csv")
print(f"\nPreview:\n{submission.head(10).to_string(index=False)}")


FINAL CROSS-VALIDATION SCORES (Stacked ensemble + OOF tier)
  Features (clf):     18
  Features (reg):     24  (+6 OOF features)
  Task A — Accuracy:  88.12%   (+/- 0.87%)
  Task B — R²:        0.8453     (+/- 0.0140)
  Task B — RMSE:      1.6417%   (+/- 0.0133%)
------------------------------------------------------------
  COMBINED (0.5·acc + 0.5·R²):  0.8633

Saved submission.csv

Preview:
 Id  RiskTier  InterestRate
  0         2          6.44
  1         0          5.76
  2         2          6.40
  3         3          6.75
  4         4         24.32
  5         3          6.60
  6         2          6.55
  7         2          6.18
  8         3          6.28
  9         2          6.37
